In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from sentence_transformers import SentenceTransformer
import faiss, glob, os
import numpy as np
import pandas as pd
import helpers
from prompts import *


# Important Paremeters

In [3]:
path = "~/kg_aug_causal_disc_exp"

In [4]:
# variables of interest
with open("variable_definitions/default_definitions.json", "r") as file:
    def_map = json.load(file)

# Markdown parsing and chunking

In [5]:
import markdown_parser
from pathlib import Path
from config import DIRECTORY

chunks = []
files = Path(DIRECTORY).glob('**/*.md')
for file in files:
    print(file)
    if os.path.isfile(file):
        # simple markdown parser that removes citations, urls, references, acknowledgements, and basically everything after the conclusion
        content = markdown_parser.process_markdown_paper(str(file))
        # semantic chunk is chunking w.r.t sentences, and has overlap param as well
        chunks.extend(markdown_parser.semantic_chunk(content) )

../clbp_causal_md/hu2022/hu2022.md
../clbp_causal_md/amiri/amiri.md
../clbp_causal_md/lancet2020/lancet2020.md
../clbp_causal_md/faravelli2013/faravelli2013.md
../clbp_causal_md/farmer2019/farmer2019.md
../clbp_causal_md/marshall2018/marshall2018.md
../clbp_causal_md/wettstein2019/wettstein2019.md
../clbp_causal_md/nida/nida.md
../clbp_causal_md/kanel2022/kanel2022.md
../clbp_causal_md/zhu2022/zhu2022.md
../clbp_causal_md/almeida/almeida.md
../clbp_causal_md/markfelder2020/markfelder2020.md
../clbp_causal_md/zhao2022/zhao2022.md
../clbp_causal_md/fluharty2017/fluharty2017.md
../clbp_causal_md/kohler2018/kohler2018.md
../clbp_causal_md/salive2013/salive2013.md
../clbp_causal_md/lenze2000/lenze2000.md
../clbp_causal_md/ohagan2023/ohagan2023.md
../clbp_causal_md/assari/assari.md
../clbp_causal_md/guan2022/guan2022.md
../clbp_causal_md/edwards2004/edwards2004.md
../clbp_causal_md/zhou2022/zhou2022.md
../clbp_causal_md/endomba2023/endomba2023.md
../clbp_causal_md/byrne2010/byrne2010.md
../c

# Creating the Retriever

In [6]:
from vllm_client import VLLMClient

In [7]:
model = SentenceTransformer('pritamdeka/S-PubMedBert-MS-MARCO')

In [8]:
embs  = model.encode(chunks, convert_to_numpy=True)

In [9]:
norms = np.linalg.norm(embs, axis=1, keepdims=True)        # shape (N, 1)
embs_normalized = embs / np.clip(norms, a_min=1e-12, a_max=None)

In [10]:
dim   = embs_normalized.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embs_normalized)

In [11]:
import numpy as np
from config import query_context_window
import helpers

# we aren't actually using k here
def get_k_docs(query: str, k: int = 100):
    # 1. Embed the query
    q_emb = model.encode([query], convert_to_numpy=True)  # shape (1, D)
    
    # 2. Search the FAISS index
    #    D: array of squared L2 distances, shape (1, k)
    #    I: array of indices of nearest neighbors, shape (1, k)
    D, I = index.search(q_emb, k)
    
    # 3. Fetch the top-k documents
    results = []
    token_count = 0
    for dist, idx in zip(D[0], I[0]):
        if helpers.token_count(chunks[idx] + "\n") + token_count > query_context_window:
            break
        token_count += helpers.token_count(chunks[idx] + "\n")
        results.append(chunks[idx]) # the original text or metadata
        

    return "\n".join(results)

In [12]:
def retrieve_context(var1, var2, debug=False):

    var1res = get_k_docs(f"{def_map.get(var1, var1)}")
    var2res = get_k_docs(f"{def_map.get(var2, var2)}")
    
    final_report = f"# Report for Variable 1: {var1}\n" + var1res + f"\n# Report for Variable 2: {var2}\n" + var2res
    if debug:
        print(final_report)
    
    return final_report

# Setting Up RAG iterators

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class Reasoning_Step(BaseModel):
    reasoning_step: str = Field(..., description="An intermediate reasoning step for breaking down the given context and query")

class Answer(BaseModel):
    reasoning: List[Reasoning_Step] = Field(..., description="List of reasoning steps")
    conclusion: bool = Field(..., description="The culminating final conclusion or answer to the question")

In [ ]:
from vllm_client import VLLMClient
generator = VLLMClient(schema=Answer)

In [ ]:
def local_retriever(query, var1, var2, summary, debug=False):
    if debug:
        print(reduce_rag(query, var1, var2, summary, def_map))
    response = generator(reduce_rag(query, var1, var2, summary, def_map), sampling_params={"n":1, "temperature":0.0, "top_k":1})

    return response.conclusion, helpers.reasoning_to_string(response)
    

In [ ]:
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, causal_lit_prompt, association_prompt

def query_local_causality(row):
    var1, var2, label = row['var1'], row['var2'], row["label"]
    # bandaid for now
    var1 = "Sleep disturbance" if var1 == "Sleep" else var1
    var2 = "Sleep disturbance" if var2 == "Sleep" else var2
        
    report = retrieve_context(var1, var2)
    pquery = plausibility_prompt(var1, var2)
    aquery = association_prompt(var1, var2)
    tquery = temporality_prompt(var1, var2)
    clquery = causal_lit_prompt(var1, var2)
    plausibility, preasoning = local_retriever(pquery, var1, var2, report)
    association, areasoning = local_retriever(aquery, var1, var2, report)
    temporality, treasoning = local_retriever(tquery, var1, var2, report)
    causal_lit, clreasoning = local_retriever(clquery, var1, var2, report)
    return [var1, var2, plausibility, preasoning, association, areasoning, temporality, treasoning, causal_lit, clreasoning, report, label]

In [ ]:
proto = pd.read_csv(f"{path}/data/proto_cleaned.csv").drop(columns=["Unnamed: 0"])
full = pd.read_csv(f"{path}/data/full_cleaned.csv").drop(columns=["Unnamed: 0"])

In [ ]:
from tqdm import tqdm

tqdm.pandas()

# Setting Up the Experiment

In [ ]:
res = proto.apply(query_local_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Plausibility", "Plausibility Reasoning", "Association", "Association Reasoning", "Temporality", "Temporality Reasoning", "Causal Literature", "Causal Literature Reasoning", "Report", "Label"
local_res = pd.DataFrame(res.to_list(), columns=columns)
local_res.to_csv("results/rag.csv")
local_res

In [ ]:
from sklearn.metrics import f1_score

f1_score(local_res["Label"], local_res["Plausibility"])

In [ ]:
f1_score(local_res["Plausibility"], local_res["Label"])

## Toy

In [ ]:
res = proto[:1].apply(query_local_causality, axis=1)

In [ ]:
columns = "Var1", "Var2", "Plausibility", "Plausibility Reasoning", "Association", "Association Reasoning", "Temporality", "Temporality Reasoning", "Causal Literature", "Causal Literature Reasoning", "Report", "Label"
local_res = pd.DataFrame(res.to_list(), columns=columns)

In [ ]:
print(local_res["Causal Literature Reasoning"].loc[0])

In [13]:
from retrieval.path_search import retrieve_path
from promptsd.query_prompts import plausibility_prompt, temporality_prompt, causal_lit_prompt, association_prompt
from prompts import reduce_rag

var1, var2 = "Education", "Depression"

pquery = plausibility_prompt(var1, var2)
report = get_k_docs(pquery)